# LeetCode 1780 — Check if Number is a Sum of Powers of Three bằng Python

Link đề: https://leetcode.com/problems/check-if-number-is-a-sum-of-powers-of-three/ · Độ khó: **Medium** · Chủ đề: Math

Skeleton LeetCode:

```python
class Solution:
    def checkPowersOfThree(self, n: int) -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu biểu diễn số trong **hệ cơ số 3** và cách lấy từng chữ số bằng `% 3` và `// 3`.
2. Thấy rằng "tổng các lũy thừa **phân biệt** của 3" ⇔ mọi chữ số cơ số 3 đều là `0` hoặc `1`.
3. Viết vét cạn mọi tập con lũy thừa (`2^15` tập) và tham lam "trừ lũy thừa lớn nhất".
4. Chứng minh tham lam đúng nhờ bất đẳng thức `3^0 + 3^1 + … + 3^{k−1} < 3^k`.
5. Kiểm thử đối chiếu trên **toàn bộ** miền nhỏ và mẫu ngẫu nhiên tới `10⁷`.

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Trả về `True` nếu có thể viết `n` thành **tổng các lũy thừa phân biệt của 3** (mỗi lũy thừa `3^x` dùng tối đa một lần), ngược lại `False`.

**Ví dụ 1:** `n = 12` → `True`: `12 = 3¹ + 3²`.

**Ví dụ 2:** `n = 91` → `True`: `91 = 3⁰ + 3² + 3⁴`.

**Ví dụ 3:** `n = 21` → `False`.

**Ràng buộc:** `1 ≤ n ≤ 10⁷`.

In [1]:
import random
import time
from itertools import combinations
from functools import lru_cache

examples = [(12, True), (91, True), (21, False)]
POWERS = [3 ** k for k in range(16) if 3 ** k <= 10 ** 7]
print("Các lũy thừa của 3 không vượt 10^7:", POWERS, "| số lượng:", len(POWERS))

Các lũy thừa của 3 không vượt 10^7: [1, 3, 9, 27, 81, 243, 729, 2187, 6561, 19683, 59049, 177147, 531441, 1594323, 4782969] | số lượng: 15


## 3. Kiến thức nền

### 3.1 Lũy thừa và vòng lặp nhân

`3 ** k` hoặc nhân dần `p *= 3`. Có 15 lũy thừa `≤ 10⁷` (`3¹⁴ = 4 782 969`, `3¹⁵ = 14 348 907`).

### 3.2 Hệ cơ số 3: `n % 3` và `n // 3`

Chữ số hàng thấp nhất của `n` trong cơ số 3 là `n % 3`; bỏ chữ số đó bằng `n // 3`. Lặp đến khi `n = 0` thu được mọi chữ số (từ thấp lên cao).

In [2]:
def to_base3(n):
    if n == 0:
        return "0"
    digits = []
    while n:
        n, r = divmod(n, 3)
        digits.append(str(r))
    return "".join(reversed(digits))

for n, _ in examples:
    print(n, "->", to_base3(n))

12 -> 110
91 -> 10101
21 -> 210


### 3.3 `int(s, 3)` — đổi chuỗi cơ số 3 về số

In [3]:
print(int("110", 3), int("10101", 3), int("210", 3))

12 91 21


### 3.4 Tổng các lũy thừa nhỏ hơn

`3⁰ + 3¹ + … + 3^{k−1} = (3^k − 1)/2 < 3^k`. Nghĩa là **mọi** lũy thừa nhỏ hơn cộng lại vẫn chưa bằng lũy thừa kế tiếp — cơ sở của tham lam.

In [4]:
for k in range(1, 7):
    print(k, sum(3 ** i for i in range(k)), "<", 3 ** k)

1 1 < 3
2 4 < 9
3 13 < 27
4 40 < 81
5 121 < 243
6 364 < 729


### 3.5 Duyệt mọi tập con bằng bitmask

Với `L` phần tử, mỗi số `mask ∈ [0, 2^L)` mã hóa một tập con: bit thứ `i` bật ⇔ chọn phần tử `i`. `mask >> i & 1` đọc bit thứ `i`. Đây là công cụ của Cách 1.

In [5]:
items = [1, 3, 9]
for mask in range(1 << len(items)):
    chosen = [items[i] for i in range(len(items)) if mask >> i & 1]
    print(f"mask={mask:03b} -> {chosen} tổng {sum(chosen)}")

mask=000 -> [] tổng 0
mask=001 -> [1] tổng 1
mask=010 -> [3] tổng 3
mask=011 -> [1, 3] tổng 4
mask=100 -> [9] tổng 9
mask=101 -> [1, 9] tổng 10
mask=110 -> [3, 9] tổng 12
mask=111 -> [1, 3, 9] tổng 13


Quan sát bảng trên: 8 tập con cho 8 tổng **khác nhau** `0, 1, 3, 4, 9, 10, 12, 13` — không có hai tập nào trùng tổng. Đó là biểu hiện của tính **duy nhất** của biểu diễn cơ số 3 với chữ số 0/1 (nếu hai tập khác nhau cùng tổng, ta có hai biểu diễn khác nhau của cùng một số).

## 4. Ý tưởng nền tảng / trực giác

Viết `n` trong cơ số 3: `n = Σ d_k · 3^k` với `d_k ∈ {0, 1, 2}` — **duy nhất**.

- `12 = 110₃` ⇒ `1·9 + 1·3 + 0·1` ⇒ dùng `3²` và `3¹` ⇒ `True`.
- `91 = 10101₃` ⇒ `3⁴ + 3² + 3⁰` ⇒ `True`.
- `21 = 210₃` ⇒ `2·9 + 1·3` — cần **hai** lần `3²` ⇒ không được dùng lặp ⇒ `False`.

Chọn tập con lũy thừa phân biệt ⇔ viết `n` với các chữ số chỉ là `0`/`1`. Vì biểu diễn cơ số 3 là duy nhất, nếu có chữ số `2` thì **không** có cách nào khác.

In [6]:
for n, exp in examples:
    b = to_base3(n)
    print(f"{n:>3} = {b}₃ -> chỉ có 0/1? {set(b) <= {'0', '1'}} | kỳ vọng {exp}")

 12 = 110₃ -> chỉ có 0/1? True | kỳ vọng True
 91 = 10101₃ -> chỉ có 0/1? True | kỳ vọng True
 21 = 210₃ -> chỉ có 0/1? False | kỳ vọng False


### 4.1 Nhìn toàn cảnh: các số hợp lệ nhỏ

Liệt kê mọi `n ≤ 40` hợp lệ cùng biểu diễn cơ số 3. Ta thấy chúng chính là các "số nhị phân" `1, 10, 11, 100, …` nhưng **đọc theo cơ số 3**: số hợp lệ thứ `k` (đếm từ 0) bằng biểu diễn nhị phân của `k` đọc như cơ số 3.

In [7]:
valid = [n for n in range(1, 41) if set(to_base3(n)) <= {"0", "1"}]
print([(n, to_base3(n)) for n in valid])
print("nhị phân của k đọc như cơ số 3:", [int(bin(k)[2:], 3) for k in range(1, len(valid) + 1)])
assert valid == [int(bin(k)[2:], 3) for k in range(1, len(valid) + 1)]

[(1, '1'), (3, '10'), (4, '11'), (9, '100'), (10, '101'), (12, '110'), (13, '111'), (27, '1000'), (28, '1001'), (30, '1010'), (31, '1011'), (36, '1100'), (37, '1101'), (39, '1110'), (40, '1111')]
nhị phân của k đọc như cơ số 3: [1, 3, 4, 9, 10, 12, 13, 27, 28, 30, 31, 36, 37, 39, 40]


Hệ quả thú vị: số lượng `n ≤ N` hợp lệ chỉ khoảng `N^{log₃2} ≈ N^{0,63}` — với `N = 10⁷` là `32 767` số (khớp với `2^15 − 1` ở Cách 4). Đa số số nguyên **không** hợp lệ.

## 5. Cách 1 — Brute Force / baseline: thử mọi tập con lũy thừa

**Ý tưởng:** chỉ có 15 lũy thừa liên quan ⇒ `2^15 = 32 768` tập con; kiểm tra có tập nào tổng bằng `n`.

In [8]:
def check_bruteforce(n):
    ps = [p for p in POWERS if p <= n]
    for mask in range(1 << len(ps)):
        if sum(ps[i] for i in range(len(ps)) if mask >> i & 1) == n:
            return True
    return False

for n, exp in examples:
    got = check_bruteforce(n); print(n, got, "kỳ vọng", exp); assert got == exp

12 True kỳ vọng True
91 True kỳ vọng True
21 False kỳ vọng False


**Độ phức tạp:** `O(2^L · L)` với `L = ⌊log₃ n⌋ + 1 ≤ 15` ⇒ ~`5·10⁵` phép mỗi `n` — chạy được cho một `n`, nhưng chậm nếu nhiều truy vấn.

**Nút thắt:** thử cả những tập chắc chắn không thể (ví dụ không chứa lũy thừa lớn nhất `≤ n` — mà như ta sẽ thấy, nó **bắt buộc** phải có).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tham lam):** gọi `3^k` là lũy thừa lớn nhất `≤ n`. Nếu không dùng `3^k`, các lũy thừa còn lại (`< 3^k`) cộng lại tối đa `(3^k − 1)/2 < 3^k ≤ n` ⇒ không đủ. Vậy **bắt buộc** dùng `3^k`, trừ đi và lặp lại với lũy thừa nhỏ hơn. Nếu sau khi trừ còn `n ≥ 3^k` (cần dùng lại `3^k`) ⇒ `False`.

**Insight 2 (cơ số 3):** lặp `n % 3`: nếu bằng `2` ⇒ `False`; ngược lại `n //= 3`. Hết vòng ⇒ `True`.

**Quy tắc:** "nếu chữ số hiện tại là 2 thì sai; nếu là 0 hoặc 1 thì bỏ chữ số đó và xét tiếp".

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam trừ lũy thừa lớn nhất

In [9]:
def check_greedy(n):
    p = 1
    while p * 3 <= n:
        p *= 3
    while p >= 1:
        if n >= p:
            n -= p
        if n >= p:                     # vẫn cần p lần nữa => phải dùng lặp
            return False
        p //= 3
    return n == 0

print([check_greedy(n) for n, _ in examples])

[True, True, False]


**Code nộp LeetCode (Cách 2):** đổi tên thành `checkPowersOfThree`. **Độ phức tạp:** `O(log₃ n)`.

### 7.2 Cách 3 — Chữ số cơ số 3

In [10]:
def check_base3(n):
    while n:
        if n % 3 == 2:
            return False
        n //= 3
    return True

print([check_base3(n) for n, _ in examples])

[True, True, False]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** Time `O(log₃ n)` ≈ 15 vòng, Space `O(1)`.

### 7.3 Cách 4 — Tiền xử lý tập mọi tổng (nhiều truy vấn)

Mọi tổng phân biệt của 15 lũy thừa là `2^15 = 32 768` số — tạo **một lần** thành `set`, mỗi truy vấn `O(1)`.

In [11]:
def build_all_sums():
    sums = [0]
    for p in POWERS:
        sums += [s + p for s in sums]
    return set(sums)

ALL_SUMS = build_all_sums()
print("số tổng khác nhau:", len(ALL_SUMS), "| 12, 91, 21 thuộc tập?", [n in ALL_SUMS for n, _ in examples])

số tổng khác nhau: 32768 | 12, 91, 21 thuộc tập? [True, True, False]


**Độ phức tạp Cách 4:** tiền xử lý `O(2^L)`, truy vấn `O(1)`, Space `O(2^L)`.

### 7.4 Cách 5 — Đệ quy theo chữ số thấp nhất (có ghi nhớ)

`ok(n) = True` nếu `n = 0`; nếu `n % 3 == 2` ⇒ `False`; ngược lại `ok(n // 3)`. Dạng đệ quy cho thấy cấu trúc "bỏ một chữ số" rõ ràng; thực chất là Cách 3.

In [12]:
@lru_cache(maxsize=None)
def check_rec(n):
    if n == 0:
        return True
    if n % 3 == 2:
        return False
    return check_rec(n // 3)

print([check_rec(n) for n, _ in examples])

[True, True, False]


## 8. TRACE từng bước

Trace Cách 3 trên ba ví dụ:

| n | n % 3 | hành động | n mới |
|---|---|---|---|
| 12 | 0 | tiếp | 4 |
| 4 | 1 | tiếp | 1 |
| 1 | 1 | tiếp | 0 ⇒ **True** |
| 21 | 0 | tiếp | 7 |
| 7 | 1 | tiếp | 2 |
| 2 | 2 | **False** | |

In [13]:
def trace_base3(n):
    k = 0; orig = n
    while n:
        r = n % 3
        print(f"  hàng 3^{k}: n={n}, n%3={r} -> {'SAI (chữ số 2)' if r == 2 else 'dùng 3^%d' % k if r == 1 else 'bỏ qua'}")
        if r == 2:
            return False
        n //= 3; k += 1
    print(f"  {orig}: mọi chữ số là 0/1 -> True")
    return True

for n, exp in examples:
    print(n); assert trace_base3(n) == exp

12
  hàng 3^0: n=12, n%3=0 -> bỏ qua
  hàng 3^1: n=4, n%3=1 -> dùng 3^1
  hàng 3^2: n=1, n%3=1 -> dùng 3^2
  12: mọi chữ số là 0/1 -> True
91
  hàng 3^0: n=91, n%3=1 -> dùng 3^0
  hàng 3^1: n=30, n%3=0 -> bỏ qua
  hàng 3^2: n=10, n%3=1 -> dùng 3^2
  hàng 3^3: n=3, n%3=0 -> bỏ qua
  hàng 3^4: n=1, n%3=1 -> dùng 3^4
  91: mọi chữ số là 0/1 -> True
21
  hàng 3^0: n=21, n%3=0 -> bỏ qua
  hàng 3^1: n=7, n%3=1 -> dùng 3^1
  hàng 3^2: n=2, n%3=2 -> SAI (chữ số 2)


Trace Cách 2 (tham lam) trên `91`:

In [14]:
def trace_greedy(n):
    p = 1
    while p * 3 <= n: p *= 3
    used = []
    while p >= 1:
        if n >= p:
            n -= p; used.append(p)
        print(f"  p={p:>3}: dùng? {used[-1] == p if used else False}, n còn {n}")
        if n >= p:
            print("  cần dùng lại p -> False"); return False
        p //= 3
    print("  dùng các lũy thừa:", used); return n == 0

assert trace_greedy(91) is True

  p= 81: dùng? True, n còn 10
  p= 27: dùng? False, n còn 10
  p=  9: dùng? True, n còn 1
  p=  3: dùng? False, n còn 1
  p=  1: dùng? True, n còn 0
  dùng các lũy thừa: [81, 9, 1]


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — chữ số cơ số 3.

In [15]:
class Solution:
    def checkPowersOfThree(self, n: int) -> bool:
        while n > 0:
            if n % 3 == 2:
                return False
            n //= 3
        return True

for n, exp in examples:
    got = Solution().checkPowersOfThree(n); print(n, got, exp); assert got == exp

12 True True
91 True True
21 False False


## 10. Vì sao thuật toán đúng

**Bổ đề (biểu diễn cơ số 3 duy nhất).** Mọi `n ≥ 0` viết được **duy nhất** dưới dạng `Σ d_k 3^k` với `d_k ∈ {0,1,2}` (định lý chia có dư áp dụng lặp lại).

**(⇐)** Nếu mọi `d_k ∈ {0,1}` thì `n = Σ_{d_k = 1} 3^k` — tổng lũy thừa phân biệt.

**(⇒)** Nếu `n = Σ_{k ∈ S} 3^k` với `S` là tập (không lặp) thì đó là một biểu diễn cơ số 3 với chữ số `0/1`; theo tính duy nhất, đó **chính là** biểu diễn cơ số 3 của `n` ⇒ không có chữ số `2`.

Vòng lặp `n % 3`, `n //= 3` liệt kê đúng các chữ số `d_0, d_1, …` ⇒ Cách 3 đúng. ∎

**Tham lam (Cách 2):** đã chứng minh ở mục 6 rằng lũy thừa lớn nhất `≤ n` là bắt buộc; nếu sau khi trừ vẫn còn `≥ p` thì `n ≥ 2p` ở hàng đó ⇔ chữ số `2`.

## 11. Phân tích Time / Space Complexity

Gọi `L = ⌊log₃ n⌋ + 1 ≤ 15`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tập con | `O(2^L · L)` | `O(L)` |
| 2. Tham lam | `O(L)` | `O(1)` |
| 3. Cơ số 3 | `O(L)` | `O(1)` |
| 4. Tập mọi tổng | tiền xử lý `O(2^L)`, truy vấn `O(1)` | `O(2^L)` |
| 5. Đệ quy | `O(L)` | `O(L)` |

In [16]:
qs = list(range(1, 200001))
for name, f in [("base3", check_base3), ("greedy", check_greedy), ("set", ALL_SUMS.__contains__)]:
    t = time.perf_counter(); cnt = sum(map(f, qs)); print(f"{name:<7} 200000 truy vấn: {cnt} True, {(time.perf_counter()-t)*1000:.0f} ms")
t = time.perf_counter(); [check_bruteforce(n) for n in range(10 ** 6, 10 ** 6 + 20)]
print(f"brute   20 truy vấn quanh 10^6: {(time.perf_counter()-t)*1000:.0f} ms")

base3   200000 truy vấn: 2789 True, 25 ms
greedy  200000 truy vấn: 2789 True, 93 ms
set     200000 truy vấn: 2789 True, 3 ms


brute   20 truy vấn quanh 10^6: 110 ms


## 12. Bộ test

In [17]:
ALL = {"greedy": check_greedy, "base3": check_base3, "set": ALL_SUMS.__contains__, "rec": check_rec,
       "submit": Solution().checkPowersOfThree}
def run_all(n):
    res = {k: f(n) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (n, res)
    return next(iter(res.values()))

for n, exp in examples:
    assert run_all(n) == exp
for n in range(1, 2000):                                     # toàn bộ miền nhỏ, có brute force
    assert run_all(n) == check_bruteforce(n), n
edge = [(1, True), (2, False), (3, True), (4, True), (5, False), (3 ** 14, True), (sum(POWERS), True),
        (sum(POWERS) + 1, False), (10 ** 7, False)]
for n, exp in edge:
    assert run_all(n) == exp, n
rng = random.Random(1780)
for _ in range(3000):
    run_all(rng.randint(1, 10 ** 7))
assert sorted(ALL_SUMS)[:8] == [0, 1, 3, 4, 9, 10, 12, 13]
print("Toàn bộ 1..1999 khớp brute force; edge case; 3000 test ngẫu nhiên tới 10^7: OK")

Toàn bộ 1..1999 khớp brute force; edge case; 3000 test ngẫu nhiên tới 10^7: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép dùng lặp lũy thừa (kiểm tra "mọi chữ số < 3")

In [18]:
def wrong_allow_repeat(n):
    while n:
        if n % 3 > 2:                           # SAI: điều kiện luôn sai, chữ số 2 lọt qua
            return False
        n //= 3
    return True

n, exp = examples[2]
print("sai :", wrong_allow_repeat(n), "| đúng:", exp)

sai : True | đúng: False


**Vì sao sai:** chữ số `2` nghĩa là cần dùng **hai lần** cùng một lũy thừa — đề yêu cầu **phân biệt**. **Sửa:** `if n % 3 == 2`.

### Lỗi 2 — Tham lam nhưng quên kiểm tra dùng lặp

In [19]:
def wrong_greedy(n):
    p = 1
    while p * 3 <= n: p *= 3
    while p >= 1:
        while n >= p:                           # SAI: trừ p nhiều lần
            n -= p
        p //= 3
    return n == 0

print("sai :", [wrong_greedy(n) for n, _ in examples], "| đúng:", [e for _, e in examples])

sai : [True, True, True] | đúng: [True, True, False]


**Vì sao sai:** trừ lặp lại cùng một `p` = dùng lũy thừa nhiều lần ⇒ mọi `n` đều về 0 (vì `3⁰ = 1`) ⇒ luôn `True`. **Sửa:** mỗi `p` trừ tối đa **một** lần và kiểm tra phần còn lại `< p`.

### Lỗi 3 — Quên `3⁰ = 1` (bắt đầu từ `3¹`)

In [20]:
def wrong_no_one(n):
    ps = [3 ** k for k in range(1, 15)]                  # SAI: thiếu 3^0
    return any(sum(c) == n for r in range(len(ps) + 1) for c in combinations([p for p in ps if p <= n], r))

n, exp = examples[1]
print("sai :", wrong_no_one(n), "| đúng:", exp)

sai : False | đúng: True


**Vì sao sai:** `91 = 3⁰ + 3² + 3⁴` cần `3⁰ = 1`. **Sửa:** lũy thừa bắt đầu từ `x = 0`.

### Lỗi 4 — Dùng `math.log` để tìm lũy thừa lớn nhất (sai số số thực)

In [21]:
import math
bad = [k for k in range(1, 20) if int(math.log(3 ** k, 3)) != k]
print("các k mà int(log(3^k, 3)) != k:", bad)
print("ví dụ: math.log(243, 3) =", repr(math.log(243, 3)))

các k mà int(log(3^k, 3)) != k: [5, 10, 13, 15, 17]
ví dụ: math.log(243, 3) = 4.999999999999999


**Vì sao sai (tiềm ẩn):** `math.log(243, 3)` có thể ra `4.999999…` ⇒ `int` cho `4`. **Sửa:** nhân dần `p *= 3` hoặc dùng cách chữ số cơ số 3 (không cần log).

### Lỗi 5 — Chỉ kiểm tra chữ số **cuối** (`n % 3`) một lần

In [22]:
def wrong_last_digit(n):
    return n % 3 != 2                                   # SAI: chỉ xét hàng 3^0

n, exp = examples[2]
print("sai :", wrong_last_digit(n), "| đúng:", exp, "| 21 =", to_base3(21), "(chữ số 2 nằm ở hàng 3^2)")

sai : True | đúng: False | 21 = 210 (chữ số 2 nằm ở hàng 3^2)


**Vì sao sai:** chữ số `2` có thể nằm ở **bất kỳ** hàng nào. **Sửa:** lặp qua mọi chữ số (`n //= 3`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng các lũy thừa **phân biệt** của `b`" ⇒ biểu diễn cơ số `b` chỉ có chữ số `0/1`.
- Với `b = 2` thì **mọi** số đều được (nhị phân) — `b ≥ 3` mới có số bị loại.
- Lũy thừa tăng nhanh ⇒ vét cạn tập con thường khả thi (≤ 2^30) nhưng không cần thiết.
- Tránh `log` cho bài số nguyên — dùng nhân/chia nguyên.

## 15. Biến thể của bài

1. **Cơ số `b` bất kỳ** (Bài tập 4).
2. **Cho phép mỗi lũy thừa dùng tối đa 2 lần** ⇒ luôn `True` (mọi số có biểu diễn cơ số 3).
3. **Cho phép cộng/trừ lũy thừa (hệ tam phân cân bằng)**: mọi số đều biểu diễn được bằng chữ số `{−1, 0, 1}` (Bài tập 5).
4. **326 Power of Three**: kiểm tra `n` có phải **một** lũy thừa của 3.
5. **Đếm các số ≤ N là tổng lũy thừa phân biệt của 3**: đếm số trong cơ số 3 chỉ gồm 0/1 — kiểu "digit DP".

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tập con | Thử `2^L` tập | `O(2^L L)` | `O(L)` | Kiểm chứng |
| 2. Tham lam | Trừ lũy thừa lớn nhất | `O(L)` | `O(1)` | Dễ giải thích |
| 3. Cơ số 3 | Không có chữ số 2 | `O(L)` | `O(1)` | **Bản nộp** |
| 4. Tập mọi tổng | Tiền xử lý | `O(1)` mỗi truy vấn | `O(2^L)` | Rất nhiều truy vấn |
| 5. Đệ quy | Bỏ chữ số thấp | `O(L)` | `O(L)` | Minh họa cấu trúc |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `n = 10`; (b) `n = 6`; (c) `n = 27`; (d) `n = 40`.

**Bài 2 (tính tay):** Đổi `n = 100` sang cơ số 3 bằng tay (chia liên tiếp) và kết luận.

**Bài 3 (điền TODO):** Hoàn thiện `powers_used(n)` trả list các lũy thừa dùng trong biểu diễn (hoặc `None` nếu không được).

**Bài 4 (tự viết lại):** Viết `check_powers(n, b)` cho cơ số `b ≥ 2` bất kỳ.

**Bài 5 (mini challenge):** Viết `balanced_ternary(n)` trả list chữ số `{−1, 0, 1}` (từ thấp lên cao) sao cho `n = Σ d_k 3^k`; kiểm tra lại bằng tổng.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def powers_used_todo(n):
    res, p = [], 1
    while n:
        # TODO: r = n % 3; nếu r == 2 trả None; nếu r == 1 thêm p vào res
        # TODO: n //= 3; p *= 3
        pass
    return res

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def check_powers(n, b):
    # TODO: như Cách 3 với cơ số b: chữ số > 1 thì False
    pass

def balanced_ternary(n):
    # TODO: r = n % 3; nếu r == 2: chữ số -1 và n = (n + 1) // 3; ngược lại chữ số r, n //= 3
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đổi sang cơ số 3. Mức 2: kiểm tra chữ số 2. Mức 3: (a) `101₃` ⇒ `True`; (b) `20₃` ⇒ `False`; (c) `1000₃` ⇒ `True`; (d) `1111₃` ⇒ `True`.

**Bài 2.** Mức 1: `100 = 3·33 + 1`. Mức 2: `33 = 3·11 + 0`, `11 = 3·3 + 2`. Mức 3: `100 = 10201₃` có chữ số 2 ⇒ `False`.

**Bài 3.** Mức 1: `r = n % 3`. Mức 2: `if r == 2: return None`. Mức 3: kiểm tra `sum(res) == n_ban_đầu`.

**Bài 4.** Mức 1: `while n`. Mức 2: `if n % b > 1: return False`. Mức 3: với `b = 2` luôn `True`.

**Bài 5.** Mức 1: chữ số 2 ≡ `3 − 1`: viết `−1` và nhớ 1 sang hàng trên. Mức 2: `n = (n + 1) // 3`. Mức 3: hàm phải dừng vì `n` giảm dần; kiểm tra với mọi `n ≤ 3000`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
b1 = [Solution().checkPowersOfThree(n) for n in [10, 6, 27, 40]]
print("Bài 1:", b1, [to_base3(n) for n in [10, 6, 27, 40]]); assert b1 == [True, False, True, True]
print("Bài 2: 100 =", to_base3(100), "->", Solution().checkPowersOfThree(100)); assert to_base3(100) == "10201"

Bài 1: [True, False, True, True] ['101', '20', '1000', '1111']
Bài 2: 100 = 10201 -> False


In [26]:
# Bài 3, 4, 5
def powers_used_todo(n):
    res, p = [], 1
    while n:
        r = n % 3
        if r == 2:
            return None
        if r == 1:
            res.append(p)
        n //= 3; p *= 3
    return res

def check_powers(n, b):
    while n:
        if n % b > 1:
            return False
        n //= b
    return True

def balanced_ternary(n):
    digits = []
    while n:
        r = n % 3
        if r == 2:
            digits.append(-1); n = (n + 1) // 3
        else:
            digits.append(r); n //= 3
    return digits

for n in range(1, 3000):
    pu = powers_used_todo(n)
    assert (pu is not None) == check_base3(n) and (pu is None or sum(pu) == n)
    for b in (2, 3, 4, 5):
        brute = any(sum(c) == n for r in range(1, 12) for c in combinations([b ** k for k in range(12) if b ** k <= n], r))
        if n < 400:
            assert check_powers(n, b) == brute
    bt = balanced_ternary(n)
    assert sum(d * 3 ** k for k, d in enumerate(bt)) == n and set(bt) <= {-1, 0, 1}
print("Bài 3, 4, 5: OK |", powers_used_todo(91), check_powers(10, 2), balanced_ternary(21))

Bài 3, 4, 5: OK | [1, 9, 81] True [0, 1, -1, 1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tổng lũy thừa phân biệt của b":**

```python
while n:
    if n % b > 1: return False
    n //= b
return True
```

**Checklist:**
- [ ] Chữ số `2` ⇒ `False` (không phải `> 2`)?
- [ ] Nhớ `3⁰ = 1`?
- [ ] Không dùng `math.log` cho số nguyên?
- [ ] Tham lam: mỗi lũy thừa trừ **tối đa một lần**?
- [ ] Đã test `1, 2, 3, 4, 5`, `3¹⁴`, tổng mọi lũy thừa, `10⁷`?